In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

class DataProcessor:
    def __init__(self):
        self.df = None
    
    def load_data(self, file_path, limit=None):
        """Load CSV or Excel file with an optional limit on rows"""
        if file_path.endswith('.csv'):
            # 'nrows' tells pandas to stop reading after X lines
            self.df = pd.read_csv(file_path, nrows=limit) 
        elif file_path.endswith(('.xlsx', '.xls')):
            self.df = pd.read_excel(file_path, nrows=limit)
        else:
            raise ValueError("Unsupported file format")
        
        print(f"✅ Loaded {len(self.df)} rows")
        return self.df
    
    def clean_data(self):
        """Remove problems from data"""
        initial_rows = len(self.df)
        
        # 1. Remove duplicate rows
        self.df = self.df.drop_duplicates()
        duplicates_removed = initial_rows - len(self.df)
        print(f"🗑️ Removed {duplicates_removed} duplicate rows")
        
        # 2. Handle missing values
        # For numeric columns, fill with median
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            missing_count = self.df[col].isnull().sum()
            if missing_count > 0:
                self.df[col].fillna(self.df[col].median(), inplace=True)
                print(f"📝 Filled {missing_count} missing values in {col}")
        
        # For categorical columns, fill with mode (most frequent)
        categorical_cols = self.df.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            if col != 'date':  # Don't fill dates
                missing_count = self.df[col].isnull().sum()
                if missing_count > 0:
                    self.df[col].fillna(self.df[col].mode()[0], inplace=True)
                    print(f"📝 Filled {missing_count} missing values in {col}")
        
        # 3. Normalize date columns
        if 'date' in self.df.columns:
            self.df['date'] = pd.to_datetime(self.df['date'], errors='coerce')
            # Remove rows with invalid dates
            self.df = self.df.dropna(subset=['date'])
        
        # 4. Remove outliers (values beyond 3 standard deviations)
        for col in numeric_cols:
            if col not in ['id', 'outlet_id', 'item_id']:  # Don't remove outliers from IDs
                mean = self.df[col].mean()
                std = self.df[col].std()
                outlier_mask = (self.df[col] < mean - 3*std) | (self.df[col] > mean + 3*std)
                outliers_removed = outlier_mask.sum()
                if outliers_removed > 0:
                    self.df = self.df[~outlier_mask]
                    print(f"🚫 Removed {outliers_removed} outliers from {col}")
        
        print(f"✅ Final dataset: {len(self.df)} rows")
        return self.df
    
    def feature_engineering(self):
        """Specific features for Food Demand Forecasting"""
        # 1. Price Difference (Very important for this dataset!)
        if 'base_price' in self.df.columns and 'checkout_price' in self.df.columns:
            self.df['discount_amount'] = self.df['base_price'] - self.df['checkout_price']
            self.df['discount_percent'] = (self.df['discount_amount'] / self.df['base_price']) * 100
            print("✅ Created discount features")

        # 2. Rolling Averages for Orders (Using 'week')
        if 'num_orders' in self.df.columns and 'week' in self.df.columns:
            # Sort by week to ensure the rolling window is chronological
            self.df = self.df.sort_values(['center_id', 'meal_id', 'week'])
            
            # Calculate rolling mean of orders for the last 4 weeks per center/meal
            self.df['rolling_orders_4wk'] = self.df.groupby(['center_id', 'meal_id'])['num_orders'].transform(
                lambda x: x.rolling(window=4, min_periods=1).mean()
            )
            print("✅ Created rolling demand features")
    
        return self.df
    
    def aggregate_by_day(self, outlet_id=None):
        """Group data by day"""
        group_cols = ['date']
        if outlet_id:
            self.df = self.df[self.df['outlet_id'] == outlet_id]
        
        agg_dict = {}
        if 'customer_count' in self.df.columns:
            agg_dict['customer_count'] = 'sum'
        if 'quantity_sold' in self.df.columns:
            agg_dict['quantity_sold'] = 'sum'
        if 'revenue' in self.df.columns:
            agg_dict['revenue'] = 'sum'
        
        daily_df = self.df.groupby(group_cols).agg(agg_dict).reset_index()
        print(f"✅ Aggregated to {len(daily_df)} daily records")
        return daily_df
    
    def save_processed_data(self, output_path):
        """Save cleaned data"""
        self.df.to_csv(output_path, index=False)
        print(f"💾 Saved to {output_path}")
        
    def merge_metadata(self, meal_path, center_path):
        """Join the sales data with meal and center descriptions"""
        if self.df is None:
            raise ValueError("Load train.csv first before merging!")

        # Load metadata files
        meal_info = pd.read_csv(meal_path)
        center_info = pd.read_csv(center_path)

        # 1. Merge Meal Info (Adds category and cuisine)
        self.df = pd.merge(self.df, meal_info, on='meal_id', how='left')
        
        # 2. Merge Center Info (Adds city and region)
        self.df = pd.merge(self.df, center_info, on='center_id', how='left')

        print(f"✅ Merged metadata. New columns: {list(self.df.columns)}")
        return self.df